# Imports

In [42]:
import numpy as np
import pandas as pd
import os

In [43]:
df_trades=pd.read_csv('../datasets/trades.csv')
df_prices=pd.read_csv('../datasets/daily_trade_prices.csv')
df_customers=pd.read_csv('../datasets/dim_customer.csv')
df_stocks=pd.read_csv('../datasets/dim_stock.csv')
df_dates=pd.read_csv('../datasets/dim_date.csv')

# Data Cleaning

### Missing Values

Each dataframe was checked for any duplicates and missing values. None had duplicates, however the prices dataset had missing values in `STK001`, `STK004`, `STK009`, `STK012`, `STK019`, `STK020`, as seen below.

In [44]:
total_duplicates = df_trades.duplicated().sum()
total_duplicates += df_prices.duplicated().sum()
total_duplicates += df_customers.duplicated().sum()
total_duplicates += df_stocks.duplicated().sum()
total_duplicates += df_dates.duplicated().sum()

print(f"Total number of duplicates: {total_duplicates}")

Total number of duplicates: 0


In [45]:
# nulls count
print("df_trades.csv null count: ")
print(df_trades.isnull().sum())
print('-----------------------')

print("df_prices.csv null count: ")
print(df_prices.isnull().sum())
print('-----------------------')

print("df_customers.csv null count: ")
print(df_customers.isnull().sum())
print('-----------------------')

print("df_stocks.csv null count: ")
print(df_stocks.isnull().sum())
print('-----------------------')

print("df_dates.csv null count: ")
print(df_dates.isnull().sum())

df_trades.csv null count: 
transaction_id                0
timestamp                     0
customer_id                   0
stock_ticker                  0
transaction_type              0
quantity                      0
average_trade_size            0
cumulative_portfolio_value    0
dtype: int64
-----------------------
df_prices.csv null count: 
date       0
STK001    42
STK002     0
STK003     0
STK004    54
STK005     0
STK006     0
STK007     0
STK008     0
STK009    57
STK010     0
STK011     0
STK012    36
STK013     0
STK014     0
STK015     0
STK016     0
STK017     0
STK018     0
STK019    54
STK020    57
dtype: int64
-----------------------
df_customers.csv null count: 
customer_key               0
customer_id                0
account_type               0
avg_trade_size_baseline    0
dtype: int64
-----------------------
df_stocks.csv null count: 
stock_key         0
stock_ticker      0
company_name      0
liquidity_tier    0
sector            0
industry          0
dtype: int64


To impute the missing values in the prices dataset, the average between the previous and next day was used.

In [46]:
# handling null prices (ONLY NULLS)
df_prices = df_prices.interpolate(method='linear', axis=0).ffill().bfill()

/tmp/ipykernel_41982/3699893189.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_prices = df_prices.interpolate(method='linear', axis=0).ffill().bfill()


### Inconsistency Detection

Each dataset then was checked for any inconsistencies. Starting with the categorical features, each value was checked to be consistently and correctly labeled.

In [47]:
print("[TRADES]")
print("\n", df_trades["stock_ticker"].value_counts(dropna=False))

print("[DAILY_TRADE_PRICES]")
print(f"Unique dates: {df_prices['date'].nunique()} / {len(df_prices)} rows")

print("[CUSTOMERS]")
print(df_customers["account_type"].value_counts(dropna=False))

print("[STOCKS]")
df_stocks["stock_ticker"].is_unique

print("[DATES]")
for col in ["month_name", "day_name", "is_weekend", "is_holiday"]:
    if col in df_dates.columns:
        print(f"\nColumn: {col}")
        print(df_dates[col].value_counts(dropna=False))

[TRADES]

 stock_ticker
STK006    2257
STK012    1656
STK018    1173
STK013    1132
STK009    1070
STK010     437
STK008     387
STK020     314
STK019     261
STK017     251
STK007     232
STK011     231
STK003     151
STK005     106
STK002      96
STK001      61
STK016      55
STK015      53
STK004      43
STK014      34
Name: count, dtype: int64
[DAILY_TRADE_PRICES]
Unique dates: 504 / 504 rows
[CUSTOMERS]
account_type
Retail           4285
Institutional     715
Name: count, dtype: int64
[STOCKS]
[DATES]

Column: month_name
month_name
May          46
January      45
August       45
October      45
March        44
July         44
November     43
April        42
September    42
June         42
February     41
December     25
Name: count, dtype: int64

Column: day_name
day_name
Monday       101
Tuesday      101
Wednesday    101
Thursday     101
Friday       100
Name: count, dtype: int64

Column: is_weekend
is_weekend
False    504
Name: count, dtype: int64

Column: is_holiday
is_holiday


Moving on to the numerical features, the features related to money were checked to be non-negative. Features related to quantity in the df_trades were checked to be positive. The foreign keys were also checked for integrity, confirming that they were valid references, while primary keys were checked for uniqueness. The `is_weekend` feature in the dates dataset was also checked to be correctly set. No inconsistencies were found. 

In [48]:

print("\n[TRADES]")
invalid_qty = df_trades[(df_trades["quantity"] <= 0) | (df_trades["average_trade_size"] <= 0)]
if not invalid_qty.empty:
    print(f"- {len(invalid_qty)} rows with invalid quantities or trade sizes")

valid_tickers = set(df_stocks["stock_ticker"])
valid_customers = set(df_customers["customer_id"])
valid_dates = set(df_dates["date"])

invalid_fk = df_trades[
    (~df_trades["stock_ticker"].isin(valid_tickers)) |
    (~df_trades["customer_id"].isin(valid_customers)) |
    (~df_trades["timestamp"].isin(valid_dates))
]
if not invalid_fk.empty:
    print(f"- {len(invalid_fk)} rows with invalid foreign keys (customer/stock/date)")

unknown_types = df_trades[~df_trades["transaction_type"].isin(["BUY", "SELL"])]
if not unknown_types.empty:
    print(f"- {len(unknown_types)} rows with invalid transaction_type values")

duplicate_trx = df_trades[df_trades.duplicated(subset=["transaction_id"], keep=False)]
if not duplicate_trx.empty:
    print(f"- {len(duplicate_trx)} duplicate transaction_id entries")

print("\n[DAILY_TRADE_PRICES]")
all_cols = set(df_prices.columns) - {"date"}
extra_cols = all_cols - set(df_stocks["stock_ticker"])
if extra_cols:
    print(f"- Columns not found: {extra_cols}")

invalid_dates_prices = df_prices[~df_prices["date"].isin(valid_dates)]
if not invalid_dates_prices.empty:
    print(f"- {len(invalid_dates_prices)} rows with invalid dates")

for col in df_prices.select_dtypes(include="number").columns:
    nonpositive = (df_prices[col] <= 0).sum()
    if nonpositive > 0:
        print(f"- Column {col} has {nonpositive} nonpositive (<=0) values")

print("\n[CUSTOMERS]")
dup_customers = df_customers[df_customers.duplicated(subset=["customer_id"], keep=False)]
if not dup_customers.empty:
    print(f"- {len(dup_customers)} duplicate customer_id entries")

print("\n[STOCKS]")
dup_stocks = df_stocks[df_stocks.duplicated(subset=["stock_ticker"], keep=False)]
if not dup_stocks.empty:
    print(f"- {len(dup_stocks)} duplicate stock_ticker entries")

print("\n[DATES]")
dup_dates = df_dates[df_dates.duplicated(subset=["date"], keep=False)]
if not dup_dates.empty:
    print(f"- {len(dup_dates)} duplicate date entries")

inconsistent_weekends = df_dates[
    (df_dates["day_name"].isin(["Saturday", "Sunday"])) & (df_dates["is_weekend"] == False)
    | (~df_dates["day_name"].isin(["Saturday", "Sunday"])) & (df_dates["is_weekend"] == True)
]
if not inconsistent_weekends.empty:
    print(f"- {len(inconsistent_weekends)} inconsistent in weekends")


[TRADES]

[DAILY_TRADE_PRICES]

[CUSTOMERS]

[STOCKS]

[DATES]


### Outlier Detection

Outlier detection was performed on each feature, with only those containing more than 10% outliers were handled.

In [49]:
def detect_outlier(df):
    for column in df.select_dtypes(include='number').columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 -  Q1

        lowerBound = Q1 - 1.5 * IQR
        upperBound = Q3 + 1.5 * IQR
        outlier = (df[column] < lowerBound) | (df[column] > upperBound)
        outlierCount = outlier.sum()
        if(outlierCount > 0):
            percentage = outlierCount/df[column].count() * 100
            percentage = round(percentage, 2)
            if(percentage > 10):
                print(f"{column} has outliers! The total percentage is: {percentage}%")
        

In [50]:
detect_outlier(df_trades)
detect_outlier(df_prices)
detect_outlier(df_customers)
detect_outlier(df_stocks)
detect_outlier(df_dates)

average_trade_size has outliers! The total percentage is: 25.25%
cumulative_portfolio_value has outliers! The total percentage is: 10.37%
STK001 has outliers! The total percentage is: 15.48%


Both `STK001` and `cumulative_portfolio_value` were easy enough to handle, as they have low percentage of outliers, so a simple log transformation was used to reduce their effect. However, since `average_trade_size` had 25% of its values seen as outliers, a simple log transformation wouldn't affect it by much. Instead, what was done is that the feature was discretized into bins according to its data summary.

In [51]:
df_prices["STK001_outlier"] = np.log(df_prices["STK001"])
df_trades["cumulative_portfolio_value_outlier"] = np.log1p(df_trades["cumulative_portfolio_value"])


In [52]:
df_trades['average_trade_size'].describe()

count    10000.000000
mean       214.254502
std        154.797352
min          8.000000
25%        176.270000
50%        201.730000
75%        212.632500
max       3401.000000
Name: average_trade_size, dtype: float64

| Label        | Bin Range (average_trade_size)        | Reason / Notes |
|---------------|--------------------------------------|----------------|
| Small         | 0 – 176.27                          | Bottom 25% of trades (low activity customers) |
| Medium        | 176.27 – 201.73                     | 25–50th percentile range (typical trade sizes) |
| Large         | 201.73 – 212.63                     | 50–75th percentile (above-average trades) |
| Very Large    | 212.63 – 500                        | 75–98th percentile (high-value trades) |
| Extreme       | 500 – max(average_trade_size)        | Isolates extreme outliers beyond typical range; prevents them from distorting analysis or model training |


In [53]:
labels = ['Small', 'Medium', 'Large', 'Very Large', 'Extreme']
bins = [0, 176.27, 201.73, 212.63, 500, df_trades['average_trade_size'].max()]
df_trades['average_trade_size_outlier'] = pd.cut(df_trades['average_trade_size'], bins=bins, labels=labels, include_lowest=True)

In [54]:
def get_outlier_percentage(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lowerBound = Q1 - 1.5 * IQR
    upperBound = Q3 + 1.5 * IQR
    outlier = (df[column] < lowerBound) | (df[column] > upperBound)
    outlierCount = outlier.sum()
    if(outlierCount > 0):
        percentage = outlierCount/df[column].count() * 100
        percentage = round(percentage, 2)
        return percentage


In [55]:
cumulative_value_outlier_pct = get_outlier_percentage(df_trades, "cumulative_portfolio_value_outlier")
stk001_outlier_pct = get_outlier_percentage(df_prices, "STK001_outlier")

print("Outlier Percentages:")
print(f" - Cumulative Portfolio Value: {cumulative_value_outlier_pct}%")
print(f" - STK001 Stock Price:         {stk001_outlier_pct}%")

print("\n------------------\n")
print("Average trade size bin counts:")
print(df_trades['average_trade_size_outlier'].value_counts())

Outlier Percentages:
 - Cumulative Portfolio Value: 0.83%
 - STK001 Stock Price:         7.54%

------------------

Average trade size bin counts:
average_trade_size_outlier
Medium        2504
Small         2501
Large         2495
Very Large    2213
Extreme        287
Name: count, dtype: int64


### Data Integration

Finally, all the datasets were merged to create the output dataframe as detailed in the Milestone 1 document.

In [56]:
def get_stock_price(row):
    stock_name = row["stock_ticker"]
    date = row["timestamp"]
    date_prices = df_prices[df_prices["date"] == date]
    stock_price = date_prices[stock_name].values[0]
    if(stock_name == "STK001"):
        stock_price = date_prices["STK001_outlier"].values[0]
        
    return stock_price

In [57]:
merged_df = df_trades.copy()
merged_df = merged_df.drop(["cumulative_portfolio_value", "cumulative_portfolio_value_outlier", "average_trade_size"], axis=1)
merged_df = merged_df.rename(columns={"average_trade_size_outlier":"average_trade_size"})

# lambda function since stock_tickers are stored as columns not rows i.e can't do a simple merge
merged_df["stock_price"] = merged_df.apply(get_stock_price, axis=1)
merged_df["total_trade_amount"] = merged_df["stock_price"] * merged_df["quantity"]

# customer id and key are exactly the same, so a join with either column works
df_customers_joinable = df_customers.copy() 
df_customers_joinable = df_customers_joinable.drop(["customer_key", "avg_trade_size_baseline"], axis=1)
df_customers_joinable = df_customers_joinable.rename(columns={"account_type":"customer_account_type"})
# there are customers with no trade inside the csv file, do we completely ignore them or add them as nulls?
merged_df = pd.merge(merged_df, df_customers_joinable, on="customer_id", how="inner")

# dates has no issues, all dates are accounted for :)
df_dates_joinable = df_dates.copy() 
df_dates_joinable = df_dates_joinable.rename(columns={"date":"timestamp"})
df_dates_joinable = df_dates_joinable.drop(["date_key","day", "month", "month_name", "quarter", "year", "day_of_week"], axis=1)
merged_df = pd.merge(merged_df, df_dates_joinable, on="timestamp", how="inner")

# stocks has no issues, all stocks are accounted for :)
df_stocks_joinable = df_stocks.copy()
df_stocks_joinable = df_stocks_joinable.drop(["stock_key", "company_name"], axis=1)
df_stocks_joinable = df_stocks_joinable.rename(columns={
    "liquidity_tier":"stock_liquidity_tier",
    "sector":"stock_sector",
    "industry":"stock_industry"
    })
merged_df = pd.merge(merged_df, df_stocks_joinable, on="stock_ticker", how="inner")

Since new features were added, they were also checked for outliers. Only `total_trade_amount` had outliers, which was handled using log transformation.

In [58]:
before = get_outlier_percentage(merged_df, "total_trade_amount")
print(f"total_trade_amount outlier percentage before handling: {before}%")

merged_df["total_trade_amount"] = np.log(merged_df["total_trade_amount"])

after = get_outlier_percentage(merged_df, "total_trade_amount")
print(f"total_trade_amount outlier percentage after handling: {after}%")

total_trade_amount outlier percentage before handling: 10.51%
total_trade_amount outlier percentage after handling: 1.15%


In [59]:
sample_df = merged_df.sample(n=10, random_state=42)
sample_df.to_csv("merged_sample.csv", index=False)
print(merged_df.shape)
sample_df.head(10)

(10000, 16)


,transaction_id,timestamp,customer_id,stock_ticker,transaction_type,quantity,average_trade_size,stock_price,total_trade_amount,customer_account_type,day_name,is_weekend,is_holiday,stock_liquidity_tier,stock_sector,stock_industry
6252,6253,2024-03-14,1006,STK006,BUY,477,Very Large,71.652430,10.439344,Retail,Thursday,False,False,High,Energy,Oil & Gas
4684,4685,2023-12-04,4955,STK006,SELL,137,Large,94.439581,9.467941,Retail,Monday,False,False,High,Energy,Oil & Gas
1731,1732,2023-05-04,696,STK012,BUY,155,Small,20.692138,8.073179,Retail,Thursday,False,False,High,Technology,Hardware
4742,4743,2023-12-06,4955,STK013,SELL,12,Large,128.352602,7.339688,Retail,Wednesday,False,False,Mid,Energy,Utilities
4521,4522,2023-11-22,1816,STK018,SELL,91,Small,232.503329,9.959764,Retail,Wednesday,False,False,Mid,Technology,Software
6340,6341,2024-03-20,1899,STK009,BUY,1509,Extreme,453.088802,13.435291,Retail,Wednesday,False,False,High,Technology,Hardware
576,577,2023-02-08,3588,STK005,BUY,106,Small,188.041313,9.900101,Retail,Wednesday,False,False,Low,Energy,Renewables
5202,5203,2024-01-03,1513,STK010,SELL,372,Medium,169.255547,11.050304,Retail,Wednesday,False,False,High,Consumer,Retail
6363,6364,2024-03-21,3588,STK019,BUY,79,Medium,51.012827,8.301525,Retail,Thursday,False,False,Mid,Energy,Oil & Gas
439,440,2023-01-30,182,STK009,BUY,19,Very Large,243.389586,8.439102,Retail,Monday,False,True,High,Technology,Hardware


### Encoding

Many of the categorical features were encoded using different techniques. Firstly, features with only two categories were encoded using binary encoding which is a subtype of label encoding. Specifically:

- `customer_account_type`: Retail: 1, Institutional: 0
- `transaction_type`: BUY: 1, SELL: 0

In [60]:
df_encoded_trans = merged_df.copy()
df_encoded_trans["transaction_type"] = df_encoded_trans["transaction_type"].map({"BUY": 1, "SELL": 0})
df_encoded_account = df_encoded_trans.copy()
df_encoded_account["customer_account_type"] = df_encoded_account["customer_account_type"].map({"Retail": 1, "Institutional": 0})


Next, columns that had more than two possible categories were instead one hot encoded, as giving them numbers would imply their order matters, which it does not.

In [61]:
df_one_hot_encoded = pd.get_dummies(
    df_encoded_account, 
    columns=["stock_ticker", "day_name", "stock_sector", "stock_industry"]
)

Finally, all true/false values were turned in 1/0, and all columns were lowercased and spaces replace with "_" for the sake of consistency.

In [62]:

one_hot_cols = [col for col in df_one_hot_encoded.columns 
            if col.startswith("stock_ticker_") 
            or col.startswith("day_name_") 
            or col.startswith("stock_sector_") 
            or col.startswith("stock_industry_")
            or col.startswith("is_")]

df_final = df_one_hot_encoded.copy()
df_final[one_hot_cols] = df_final[one_hot_cols].astype(int)
df_final.rename(columns={col: col.lower().replace(" ", "_") for col in one_hot_cols}, inplace=True)
print(f"The final columns are: {df_final.columns}")


The final columns are: Index(['transaction_id', 'timestamp', 'customer_id', 'transaction_type',
       'quantity', 'average_trade_size', 'stock_price', 'total_trade_amount',
       'customer_account_type', 'is_weekend', 'is_holiday',
       'stock_liquidity_tier', 'stock_ticker_stk001', 'stock_ticker_stk002',
       'stock_ticker_stk003', 'stock_ticker_stk004', 'stock_ticker_stk005',
       'stock_ticker_stk006', 'stock_ticker_stk007', 'stock_ticker_stk008',
       'stock_ticker_stk009', 'stock_ticker_stk010', 'stock_ticker_stk011',
       'stock_ticker_stk012', 'stock_ticker_stk013', 'stock_ticker_stk014',
       'stock_ticker_stk015', 'stock_ticker_stk016', 'stock_ticker_stk017',
       'stock_ticker_stk018', 'stock_ticker_stk019', 'stock_ticker_stk020',
       'day_name_friday', 'day_name_monday', 'day_name_thursday',
       'day_name_tuesday', 'day_name_wednesday', 'stock_sector_consumer',
       'stock_sector_energy', 'stock_sector_finance',
       'stock_sector_healthcare', 'stoc